<a href="https://colab.research.google.com/github/siaewjojwafo/26_AI/blob/main/notebooks/s01_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1회차 실습 노트북

실습가이드의 명령을 순서대로 셀로 옮긴 노트북임. 코랩, 개인 PC, 실습 서버에서 똑같이 쓸 수 있음

- 셀은 위에서부터 차례로 Shift + Enter로 실행함
- `%%writefile` 셀은 빈칸 `____`을 채운 뒤 실행하고, 바로 아래 셀로 결과를 봄
- 연결이 끊기면 0번 셀부터 다시 실행함
- 작업을 남기려면 파일 → Drive에 사본 저장. 결과 파일은 맨 아래 셀로 내려받음

## 0. 준비
- 런타임 유형은 CPU면 됨
- 아래 두 셀을 실행해 판정이 '통과'인지 봄. '확인 필요'가 나오면 TA를 부름

In [ ]:
import os, sys, shutil, subprocess, threading
from IPython.display import Image, Markdown, display

REPO = "https://github.com/elliediaz/llm26-course.git"
if os.path.isfile("../src/config.py"):          # 저장소 안에서 열었을 때
    HOME = os.path.abspath("..")
elif os.path.isdir("/workspace/course"):        # 실습 서버
    HOME = "/workspace/course"
else:                                           # 코랩
    HOME = "/content/course" if os.path.isdir("/content") else os.path.abspath("llm26-course")
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, HOME], check=True)
    else:
        subprocess.run(["git", "-C", HOME, "pull", "-q"])
os.chdir(HOME)
print("작업 폴더:", os.getcwd())
os.makedirs("results/fig", exist_ok=True)


def show(*paths, width=760):
    for p in paths:
        display(Image(p, width=width)) if os.path.exists(p) else print("아직 없음:", p)


def tail(path, n=8):
    if not os.path.exists(path):
        print("아직 없음:", path)
        return
    lines = open(path, encoding="utf-8", errors="replace").read().splitlines()
    print("\n".join(lines[-n:]))


def use_ref():
    # 학습이 늦을 때 참고 결과를 복사함(내 결과는 덮어쓰지 않음)
    n = 0
    for f in sorted(os.listdir("ref/s01")):
        dst = {".json": "results", ".png": "results/fig", ".pt": "runs"}.get(os.path.splitext(f)[1])
        if dst and not os.path.exists(os.path.join(dst, f)):
            os.makedirs(dst, exist_ok=True)
            shutil.copy(os.path.join("ref/s01", f), dst)
            n += 1
    print(f"참고 결과 {n}개 복사(이미 있던 파일은 그대로 둠)")

In [ ]:
%pip install -q koreanize-matplotlib sentence-transformers
!python src/check_gpu.py

## 실습 A. Dense로 이탈(분류)과 플레이 시간(회귀)
- 데이터: 가상 RPG 유저 2만 명의 활동 12개(`data/G/churn.csv`)
- 따라 치기 T1: 빈칸 6개(답은 실습가이드 실습 A)

In [ ]:
%%writefile typing/s01/t1_dense_heads.py
"""[따라 치기 T1] 몸통은 같고 출력층만 다르다: Dense 신경망으로 분류와 회귀 (1회차 2교시 실습 A)

실행: python typing/s01/t1_dense_heads.py      (빈칸 ____을 채운 뒤)
확인: ① 두 모델의 층별 출력 모양이 마지막 줄만 다름: 분류 (N,2), 회귀 (N,1)
      ② 분류 손실(교차 엔트로피)과 회귀 손실(평균제곱오차)이 각각 숫자 하나로 나옴
      ③ 분류 출력에 softmax를 씌우면 두 칸의 합이 1(이탈·유지 확률)
"""
import sys

import torch
import torch.nn as nn

sys.path.insert(0, "src")
import nn_tools as T  # noqa: E402

N_IN = 12                                   # 유저 활동 지표 12개(data/G/churn.csv)
x = torch.randn(8, N_IN)                    # 유저 8명 묶음(실습 A 에서는 실제 데이터)


def body():
    return nn.Sequential(nn.Linear(____, 64), nn.____(), nn.Linear(64, 32), nn.ReLU())


cls_model = nn.Sequential(body(), nn.Linear(32, ____))   # 분류: 이탈·유지 두 칸의 점수
reg_model = nn.Sequential(body(), nn.Linear(32, ____))   # 회귀: 다음 주 플레이 시간 숫자 하나

print("[분류 모델]")
T.shape_trace(cls_model, x)
print("[회귀 모델]")
T.shape_trace(reg_model, x)

y_cls = torch.tensor([0, 1, 0, 0, 1, 0, 0, 0])                 # 이탈 여부(정답 클래스 번호)
y_reg = torch.tensor([42.0, 3.0, 55.0, 18.0, 0.0, 61.0, 35.0, 27.0])  # 다음 주 일평균 플레이 분

loss_cls = nn.____()(cls_model(x), y_cls)
loss_reg = nn.____()(reg_model(x).squeeze(1), y_reg)
print("분류 손실(교차 엔트로피):", round(loss_cls.item(), 3))
print("회귀 손실(평균제곱오차):", round(loss_reg.item(), 1))

p = torch.softmax(cls_model(x), dim=1)
print("첫 유저의 [유지, 이탈] 확률:", [round(v, 3) for v in p[0].tolist()], "합", round(p[0].sum().item(), 3))

In [ ]:
!python typing/s01/t1_dense_heads.py

In [ ]:
!python src/nn_dense.py
show("results/fig/nn_dense.png")

확인 값
- 모양표 마지막 줄: 분류 `[N,2]`, 회귀 `[N,1]`
- 분류 정확도 약 0.83(기준선 0.72), 회귀 평균 오차 약 19분(기준선 40분)

## 실습 B. CNN으로 옷 사진 10종
- 데이터: Fashion-MNIST 옷 사진 7만 장(28×28 흑백)
- 따라 치기 T2: 빈칸 6개

In [ ]:
%%writefile typing/s01/t2_cnn_shapes.py
"""[따라 치기 T2] CNN 안에서 이미지 모양이 어떻게 바뀌나 (1회차 2교시 실습 B)

실행: python typing/s01/t2_cnn_shapes.py      (빈칸 ____을 채운 뒤)
확인: ① [N,1,28,28] → [N,16,28,28] → [N,16,14,14] → [N,32,14,14] → [N,32,7,7] → [N,1568] → [N,10]
      ② 펼치기 직전 크기 32×7×7 = 1568이 Linear의 입력 크기와 같아야 오류가 나지 않음
      ③ 합성곱 층의 파라미터(160, 4,640)가 Linear 층(200,832)보다 훨씬 적음: 작은 필터를 이미지 전체에 재사용하기 때문
"""
import sys

import torch
import torch.nn as nn

sys.path.insert(0, "src")
import nn_tools as T  # noqa: E402

x = torch.rand(4, 1, 28, 28)                              # 흑백 옷 사진 4장(Fashion-MNIST 크기)

model = nn.Sequential(
    nn.Conv2d(____, 16, kernel_size=3, padding=1),           # 3×3 필터 16장 → 무늬 지도 16장
    nn.ReLU(),
    nn.____(2),                                      # 가로·세로 절반
    nn.Conv2d(____, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),                                         # [N,32,7,7] → [N,1568]
    nn.Linear(32 * ____ * ____, 128),
    nn.ReLU(),
    nn.Linear(128, ____),                                   # 10종 중 하나: 분류
)
T.shape_trace(model, x)

In [ ]:
!python typing/s01/t2_cnn_shapes.py

In [ ]:
!python src/nn_cnn.py
show("results/fig/nn_cnn_pred.png", "results/fig/nn_cnn_maps.png")

확인 값
- 정확도 MLP 약 0.84, CNN 약 0.88
- 가장 많이 헷갈린 쌍: 셔츠와 티셔츠

## 해보기 ①
- `tasks/s01_해보기.md`에서 골라 값을 하나만 바꿔 실행하고 기록표에 남김

In [ ]:
!python src/nn_dense.py --hidden 8
# !python src/nn_cnn.py --target group

## 휴식 전 걸어 두기
- 실습 C·D에 쓸 학습을 뒤에서 돌림(약 7분)
- 쉬는 동안 브라우저 탭을 닫지 않음. 도는 동안 다른 학습 셀은 실행하지 않음

In [ ]:
JOBS = [(["src/nn_rnn.py"], "results/log_nn_rnn.txt"),
        (["src/tiny_seq2seq_train.py", "--arch", "all", "--minutes", "2"], "results/log_seq2seq.txt")]


def _run_jobs():
    for args, log in JOBS:
        with open(log, "w", encoding="utf-8") as f:
            subprocess.run([sys.executable, *args], stdout=f, stderr=subprocess.STDOUT)
    open("results/bg_done.txt", "w").write("done")


if os.path.exists("results/bg_done.txt"):
    os.remove("results/bg_done.txt")
threading.Thread(target=_run_jobs, daemon=True).start()
print("뒤에서 학습 시작. 진행 확인은 다음 셀")

In [ ]:
tail("results/log_nn_rnn.txt", 4); print("---"); tail("results/log_seq2seq.txt", 4)
print("\n모두 끝남" if os.path.exists("results/bg_done.txt") else "\n아직 학습 중")

## 실습 C. RNN과 LSTM, 내 문장 감성 분류
- 데이터: 네이버 영화 리뷰 4만 건(`data/common/nsmc/`)
- 학습이 덜 끝났으면 `use_ref()`로 참고 결과를 받아 진행함

In [ ]:
tail("results/log_nn_rnn.txt", 30)
# use_ref()

In [ ]:
!python src/nn_rnn.py --cell lstm --predict "배우들 연기는 좋았는데 결말이 너무 허무하다" "인생 영화 등극 ㅠㅠ 두 번 봤어요"

확인 값
- 정확도 RNN 약 0.73, LSTM 약 0.80(81자 이상 리뷰에서 차이가 가장 큼)
- 따옴표 안의 문장을 내 문장으로 바꿔 틀리는 예를 찾아봄

## 트랜스포머 이전 구조의 한계(시연)
- 첫 숫자 기억하기(찍으면 0.1), VAE(사진을 숫자 몇 개로 줄였다가 다시 그림)

In [ ]:
!python src/seq_limits.py      # 코랩 CPU에서 약 1분 30초

In [ ]:
!python src/nn_vae.py
show("results/fig/nn_vae_recon.png", "results/fig/nn_vae_latent.png")

## 실습 D. seq2seq와 어텐션
- 한글 금액을 숫자로 바꾸기(예: '삼억 칠천이백만 오천 원' → 372,005,000)

In [ ]:
tail("results/log_seq2seq.txt", 6)
!python src/viz_attention.py tiny --arch gru_attn --text "삼억 칠천이백만 오천 원"
show("results/fig/attn_tiny_gru_attn.png")

확인 값
- 17자 이상에서 `gru`는 떨어지고(코랩 약 0.56), `gru_attn`은 약 1.00

## 실습 E. 디코더 LLM은 다음 토큰 분류기
- 따라 치기 T3: 빈칸 5개
- 처음 실행할 때 모델을 내려받아 1~2분 더 걸림

In [ ]:
%%writefile typing/s01/t3_next_token.py
"""[따라 치기 T3] 다음 토큰 분류기로서의 LLM (1회차 3교시 실습 E)

실행: python typing/s01/t3_next_token.py      (빈칸 ____을 채운 뒤)
확인: ① 출력층 크기 = 어휘 크기(약 15만) = 클래스 수
      ② 다음 토큰 확률 5개, 확률 합 = 1
      ③ 생성 = 분류를 15번 반복한 결과
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEV == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE).to(DEV)   # T4 등 BF16 미지원 GPU는 FP16

# 1) 출력층의 크기 = 고를 수 있는 '클래스'의 수
print("어휘 크기(클래스 수):", model.config.____)
print("출력층:", model.lm_head)

# 2) 분류 한 번: 마지막 위치에서 다음 토큰의 확률 분포
ids = tok("결제했는데 아이템이 안", return_tensors="pt").to(DEV)
with torch.no_grad():
    logits = model(**ids).logits                      # (1, 입력 길이, 어휘 크기)
probs = torch.____(logits[0, -1].float(), dim=-1)
top = torch.____(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{p.item():.3f}  {tok.decode(i)!r}")
print("확률 합:", round(probs.sum().item(), 4))

# 3) 생성 = 분류를 반복: 고른 토큰을 뒤에 붙이고 다시 분류
x = ids["input_ids"]
for _ in range(15):
    with torch.no_grad():
        nxt = model(x).logits[0, -1].____()        # 가장 확률이 높은 클래스(greedy)
    x = torch.____([x, nxt.view(1, 1)], dim=1)
print(tok.decode(x[0]))

In [ ]:
!python typing/s01/t3_next_token.py

In [ ]:
!python src/seq2seq_demo.py
if os.path.exists("results/structure_compare.md"):
    display(Markdown(open("results/structure_compare.md", encoding="utf-8").read()))

확인 값
- 어휘 크기 151,936, 확률 합 1.0

## 해보기 ② D. 내 질문 10건으로 FAQ 검색기 재기
- 아래 셀의 질문 10건을 고침(표현 바꾸기 5, 오타 2, 은어 2, 무관 질문 1)
- `gold_faq_id`에 정답 번호를 씀(`data/G/faq.json`). 무관 질문은 `null`
- 10건 중 8건 이상 맞히면 완료

In [ ]:
%%writefile my_queries.json
[
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "오타",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "오타",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "은어",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "은어",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "무관 질문",
  "query": "여기에 FAQ와 상관없는 질문을 씀",
  "gold_faq_id": null
 }
]

In [ ]:
!python src/faq_search.py --queries my_queries.json

- 선택: 접속자 예측에서 기준선 이기기

In [ ]:
!python src/nn_rnn.py --task ccu

## 결과 내려받기
- 코랩은 연결이 끝나면 파일이 사라지므로 수업을 마치기 전에 실행함

In [ ]:
shutil.make_archive("s01_results", "zip", ".", "results")
try:
    from google.colab import files
    files.download("s01_results.zip")
except ImportError:
    print("저장 위치:", os.path.abspath("s01_results.zip"))

## 문제가 생기면
| 증상 | 조치 |
|---|---|
| `No such file or directory` | 0번 셀부터 다시 실행 |
| 연결이 끊김 | 다시 연결하고 0번 셀부터 실행, 필요하면 `use_ref()` |
| 셀이 끝나지 않음 | 셀 왼쪽 ■를 누르고 TA를 부름 |
| `Rate limited` 문구 | 1~2분 기다리면 이어짐 |